In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import sys

sys.path.append("..")

from component.script.dataset import Dataset
from component.script.project import Project


## Set user parameters

In [ ]:
project_name = "nuevo3"

# Load the project from JSON
project = Project.load(project_name=project_name)


# Define dataset

In [ ]:
dataset = Dataset(project=project)


In [ ]:
# if not parameters passed, the function will show all the available variables
dataset.set_target()


In [ ]:
dataset.set_target("forest_loss_2015_2020")


In [ ]:
dataset.set_year(2020)  # This will set the year for time-dependent variables
dataset.set_features(
    [
        "towns",
        "centros_poblados_dist",
        "forest_gfc",
        "protected_area",
        "rivers",
        "roads",
        "slope",
        "subj",
    ]
)


In [ ]:
dataset.validate()


In [ ]:
from component.script.sampling import Sampling, SamplingStrategy

sampling = Sampling(
    strategy=SamplingStrategy.legacy,
    n_samples=10000,
    seed=33,  # for reproducibility
    adapt=True,
    pixel_area_ha=0.09,
)


In [ ]:
model_identifier_name = "v1"
random_seed = 1


In [ ]:
dataset.show()


In [ ]:
samples_v1 = dataset.to_dataframe(sampling=sampling)
samples_v1


In [ ]:
print(samples_v1["towns"].unique())


## Training formula


In [ ]:
from component.script.far_helpers import generate_patsy_formula


In [ ]:
# user_formula = "I(1-fcc) + trial ~ scale(altitude) + scale(dist_edge) + scale(dist_river) + scale(dist_road) + scale(dist_town) + scale(slope) + C(pa)"
user_formula = None


In [ ]:
calculated_formula = generate_patsy_formula(dataset)


In [ ]:
if user_formula is None:
    training_formula = calculated_formula
elif user_formula is not None:
    training_formula = user_formula
training_formula


## Train glm based on period

In [ ]:
from component.script import GLMModel, ICARModel, MWModel, RFModel

# model = GLMModel(
#     name="glm_v1", dataset=dataset, sampling=sampling, random_seed=random_seed
# )
# model = RFModel(
#     name="rf_v1", dataset=dataset, sampling=sampling, random_seed=random_seed
# )
# model = ICARModel(
#     name="icarv1", dataset=dataset, sampling=sampling, random_seed=random_seed
# )
model = MWModel(
    name="mwmodel", dataset=dataset, sampling=sampling, random_seed=random_seed
)
model.register(project)
model.fit()


In [ ]:
paths = dataset.get_file_paths()
forest_gfc_path = paths["forest_gfc"]
forest_gfc_path


In [ ]:
# output_file = project.folders.glm_model_folder / model.name / "glm_calibration5.tif"
output_file = project.folders.rf_model / model.name / "rf_calibration1.tif"

model.apply(output_file, dataset, forest_gfc_path, 0)


In [ ]:
print(model.name)


In [ ]:
# Predict over historical period

period_c = "forecast"


model = get_trained_model(
    period_dictionaries, period_c, f"glm_model_{model_identifier_name}.pickle"
)

glm_predict_forecast = apply_glm_period(
    period_dictionaries,
    period_c,
    model,
)


In [ ]:
print("Done!")
